## Spark Streaming Read from Sockets | Convert Batch Code to Streaming Code

In [ ]:
# Generate Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Reading from Sockets")
    .master("local[*]")
    .getOrCreate()
)

spark

spark.conf.set("spark.sql.shuffle.partitions", 8)  # reduce number of partitions for easier viewing

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/21 02:33:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


25/12/21 02:33:20 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [4]:
# Read input data

df_raw = spark.read.format("text").load("input/example.txt")
df_raw.printSchema()

root
 |-- value: string (nullable = true)



In [5]:
df_raw.show()

+--------------------+
|               value|
+--------------------+
|simon had a dog a...|
+--------------------+



In [6]:
# Split the line into words
from pyspark.sql.functions import split

df_words = df_raw.withColumn("words", split(df_raw.value, " "))
df_words.show(truncate=False)

+------------------------------------------------------------+----------------------------------------------------------------------------+
|value                                                       |words                                                                       |
+------------------------------------------------------------+----------------------------------------------------------------------------+
|simon had a dog and a cat the dog and cat used to love simon|[simon, had, a, dog, and, a, cat, the, dog, and, cat, used, to, love, simon]|
+------------------------------------------------------------+----------------------------------------------------------------------------+



In [8]:
# Explode the list of words
from pyspark.sql.functions import explode

df_explode = df_words.withColumn("word", explode("words")).drop("value","words")
df_explode.show(truncate=False)

+-----+
|word |
+-----+
|simon|
|had  |
|a    |
|dog  |
|and  |
|a    |
|cat  |
|the  |
|dog  |
|and  |
|cat  |
|used |
|to   |
|love |
|simon|
+-----+



In [9]:
# Aggregate the words to generate count
from pyspark.sql.functions import count, lit

df_agg = df_explode.groupBy("word").agg(count(lit(1)).alias("count"))
df_agg.show(truncate=False)

+-----+-----+
|word |count|
+-----+-----+
|used |1    |
|simon|2    |
|dog  |2    |
|love |1    |
|had  |1    |
|cat  |2    |
|the  |1    |
|and  |2    |
|a    |2    |
|to   |1    |
+-----+-----+



In [10]:
# Read input data

df_raw = spark.readStream.format("socket").option("host","localhost").option("port", "9999").load()
df_raw.printSchema()

root
 |-- value: string (nullable = true)



25/12/21 03:37:31 WARN TextSocketSourceProvider: The socket source should not be used for production applications! It does not support recovery.


In [11]:
# Split the line into words
from pyspark.sql.functions import split

df_words = df_raw.withColumn("words", split("value", " "))

In [12]:
# Explode the list of words
from pyspark.sql.functions import explode

df_explode = df_words.withColumn("word", explode("words")).drop("value", "words")

In [13]:
# Aggregate the words to generate count
from pyspark.sql.functions import count, lit

df_agg = df_explode.groupBy("word").agg(count(lit(1)).alias("cnt"))

In [ ]:
# Write the output to console streaming

df_agg.writeStream.format("console").outputMode("complete").start().awaitTermination() # Update and Append are two other modes

25/12/21 03:41:18 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-1b44096b-1725-4d75-b5f0-86463524289d. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/12/21 03:41:18 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+----+---+
|word|cnt|
+----+---+
+----+---+



-------------------------------------------
Batch: 1
-------------------------------------------
+-----+---+
| word|cnt|
+-----+---+
|hello|  1|
|world|  1|
+-----+---+



-------------------------------------------
Batch: 2
-------------------------------------------
+-----+---+
| word|cnt|
+-----+---+
|hello|  1|
|world|  2|
+-----+---+



-------------------------------------------
Batch: 3
-------------------------------------------
+-----+---+
| word|cnt|
+-----+---+
|hello|  1|
|   is|  1|
| very|  1|
|world|  3|
|    a|  1|
| this|  1|
|  big|  1|
+-----+---+



25/12/21 03:43:15 WARN TextSocketMicroBatchStream: Stream closed by localhost:9999
ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

25/12/21 12:10:13 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 26670257 ms exceeds timeout 120000 ms
25/12/21 12:10:13 WARN SparkContext: Killing executors is not supported by current scheduler.
25/12/21 12:10:14 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint

In [ ]:
# Close the Spark session
spark.stop()

ConnectionRefusedError: [Errno 111] Connection refused

Streaming output modes<br />
1. Complete
2. Update
3. Append
<br />
## Spark Streaming Read from Files | Flatten JSON data

In [1]:
# Create the Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Streaming Process Files")
    .config("spark.streaming.stopGracefullyOnShutdown", True)
    .master("local[*]")
    .getOrCreate()
)

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/08 15:24:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# Create the streaming_df to read from input directory
streaming_df = (
	spark.read
	.format("json")
    .option("multiline", "true")
    .load("./input/device_files/")
)

In [ ]:
# To the schema of the data, place a sample json file and change readStream to read
streaming_df.printSchema()
# streaming_df.show(truncate=False)

root
 |-- customerId: string (nullable = true)
 |-- data: struct (nullable = true)
 |    |-- devices: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- deviceId: string (nullable = true)
 |    |    |    |-- measure: string (nullable = true)
 |    |    |    |-- status: string (nullable = true)
 |    |    |    |-- temperature: long (nullable = true)
 |-- eventId: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)



26/01/08 15:24:55 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [4]:
# Lets explode the data as devices contains list/array of device reading
from pyspark.sql.functions import explode

exploded_df = streaming_df.withColumn("data_devices", explode("data.devices"))
exploded_df.show(truncate=False)

+----------+-------------------------------------------------------------------------+------------------------------------+-----------+--------------+--------------------------+----------------------+
|customerId|data                                                                     |eventId                             |eventOffset|eventPublisher|eventTime                 |data_devices          |
+----------+-------------------------------------------------------------------------+------------------------------------+-----------+--------------+--------------------------+----------------------+
|CI00101   |{[{D004, C, SUCCESS, 20}, {D004, C, SUCCESS, 1}, {D002, C, SUCCESS, 21}]}|1450324a-c546-4175-a6d8-ee58822e1d41|10038      |device        |2023-01-05 11:13:53.650313|{D004, C, SUCCESS, 20}|
|CI00101   |{[{D004, C, SUCCESS, 20}, {D004, C, SUCCESS, 1}, {D002, C, SUCCESS, 21}]}|1450324a-c546-4175-a6d8-ee58822e1d41|10038      |device        |2023-01-05 11:13:53.650313|{D004, C, SUCCESS, 

In [5]:
# Flatten the exploded df
from pyspark.sql.functions import col

flattened_df = (
    exploded_df
    .drop("data")
    .withColumn("deviceId", col("data_devices.deviceId"))
    .withColumn("measure", col("data_devices.measure"))
    .withColumn("status", col("data_devices.status"))
    .withColumn("temperature", col("data_devices.temperature"))
    .drop("data_devices")
)

# Check the schema of the flattened_df, place a sample json file and change readStream to read
# flattened_df.printSchema()
flattened_df.show(truncate=False)

+----------+------------------------------------+-----------+--------------+--------------------------+--------+-------+-------+-----------+
|customerId|eventId                             |eventOffset|eventPublisher|eventTime                 |deviceId|measure|status |temperature|
+----------+------------------------------------+-----------+--------------+--------------------------+--------+-------+-------+-----------+
|CI00101   |1450324a-c546-4175-a6d8-ee58822e1d41|10038      |device        |2023-01-05 11:13:53.650313|D004    |C      |SUCCESS|20         |
|CI00101   |1450324a-c546-4175-a6d8-ee58822e1d41|10038      |device        |2023-01-05 11:13:53.650313|D004    |C      |SUCCESS|1          |
|CI00101   |1450324a-c546-4175-a6d8-ee58822e1d41|10038      |device        |2023-01-05 11:13:53.650313|D002    |C      |SUCCESS|21         |
+----------+------------------------------------+-----------+--------------+--------------------------+--------+-------+-------+-----------+



In [6]:
# To allow automatic schemaInference while reading
spark.conf.set("spark.sql.streaming.schemaInference", True)

# Create the streaming_df to read from input directory
streaming_df = (
    spark
    .readStream
    .option("multiline", "true")
    .option("cleanSource", "archive")
    .option("sourceArchiveDir", "archive_dir")
    .option("maxFilesPerTrigger", 1)
    .format("json")
    .load("./input/device_files/")
)

In [7]:
# To the schema of the data, place a sample json file and change readStream to read
streaming_df.printSchema()
# streaming_df.show(truncate=False)

root
 |-- customerId: string (nullable = true)
 |-- data: struct (nullable = true)
 |    |-- devices: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- deviceId: string (nullable = true)
 |    |    |    |-- measure: string (nullable = true)
 |    |    |    |-- status: string (nullable = true)
 |    |    |    |-- temperature: long (nullable = true)
 |-- eventId: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)



In [8]:
# Lets explode the data as devices contains list/array of device reading
from pyspark.sql.functions import explode

exploded_df = streaming_df.withColumn("data_devices", explode("data.devices"))
# exploded_df.show(truncate=False)
exploded_df.printSchema()

root
 |-- customerId: string (nullable = true)
 |-- data: struct (nullable = true)
 |    |-- devices: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- deviceId: string (nullable = true)
 |    |    |    |-- measure: string (nullable = true)
 |    |    |    |-- status: string (nullable = true)
 |    |    |    |-- temperature: long (nullable = true)
 |-- eventId: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)
 |-- data_devices: struct (nullable = true)
 |    |-- deviceId: string (nullable = true)
 |    |-- measure: string (nullable = true)
 |    |-- status: string (nullable = true)
 |    |-- temperature: long (nullable = true)



In [9]:
# Flatten the exploded df
from pyspark.sql.functions import col

flattened_df = (
    exploded_df
    .drop("data")
    .withColumn("deviceId", col("data_devices.deviceId"))
    .withColumn("measure", col("data_devices.measure"))
    .withColumn("status", col("data_devices.status"))
    .withColumn("temperature", col("data_devices.temperature"))
    .drop("data_devices")
)

# Check the schema of the flattened_df, place a sample json file and change readStream to read
flattened_df.printSchema()
# flattened_df.show(truncate=False)

root
 |-- customerId: string (nullable = true)
 |-- eventId: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)
 |-- deviceId: string (nullable = true)
 |-- measure: string (nullable = true)
 |-- status: string (nullable = true)
 |-- temperature: long (nullable = true)



In [ ]:
# Write the output to console sink to check the output

(flattened_df
 .writeStream
 .format("csv")
#  .format("console")
 .outputMode("append")
 .option("path", "./output/device_data.csv")
 .option("checkpointLocation", "checkpoint_dir")
 .start()
 .awaitTermination())

In [26]:
# Close the Spark session
spark.stop()

## Spark Streaming Read from Kafka | Real time streaming from Kafka

In [1]:
# Create the Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Streaming from Kafka")
    .config("spark.streaming.stopGracefullyOnShutdown", True)
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0')
    .config("spark.sql.shuffle.partitions", 4)
    .master("local[*]")
    .getOrCreate()
)

spark

:: loading settings :: url = jar:file:/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1f4ede0c-58c7-40d0-9cdd-2c592993e94a;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.3.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.3.0 in central
	found org.apache.kafka#kafka-clients;2.8.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.8.4 in central
	found org.slf4j#slf4j-api;1.7.32 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.2 in central
	found org.spark-project.spark#unused;1.0.0 in central
	found org.apache.hadoop#hadoop-client-api;3.3.2 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
downloading https://r

### Commands used
docker exec -it cont_d bash<br />
kafka-topics --list --bootstrap-server localhost:19092<br />
kafka-topics --create --topic device-data --bootstrap-server localhost:19092<br />
kafka-console-producer --topic device-data --bootstrap-server localhost:19092<br />

In [ ]:
# Create the kafka_df to read from kafka

kafka_df = (
    spark
    # .read
    .readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:19092")
    .option("subscribe", "device-data")
    .option("startingOffsets", "earliest")
    .load()
)

In [ ]:
# View schema for raw kafka_df
kafka_df.printSchema()
# kafka_df.show() # show won't work on streaming data, it will work when the spark session is opened in read mode

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [ ]:
(
    kafka_df
    .writeStream
    .format("console")
    .outputMode("append")            # usually append for kafka sources
    .option("truncate", False)
    .start()
    .awaitTermination(5)            # block for 5 seconds, I am closing the session after 5 seconds for demo purpose, ideally it should run indefinitely to keep processing incoming data
)

26/01/09 14:02:39 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-f98f2f5c-4970-48a7-b0c1-7b1ee5271fd5. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/01/09 14:02:39 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/01/09 14:02:39 WARN AdminClientConfig: The configuration 'key.deserializer' was supplied but isn't a known config.
26/01/09 14:02:39 WARN AdminClientConfig: The configuration 'value.deserializer' was supplied but isn't a known config.
26/01/09 14:02:39 WARN AdminClientConfig: The configuration 'enable.auto.commit' was supplied but isn't a known config.
26/01/09 14:02:39 WARN AdminClientConfig: The configuration 'max.poll.records' was supplied but isn't a known con

-------------------------------------------
Batch: 0
-------------------------------------------
+----+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+------+-----------------------+-------------+
|key |value         

False

In [ ]:
# Parse value from binay to string into kafka_json_df
from pyspark.sql.functions import expr

kafka_json_df = kafka_df.withColumn("value", expr("cast(value as string)"))
(
    kafka_json_df
    .writeStream
    .format("console")
    .outputMode("append")            # usually append for kafka sources
    .option("truncate", False)
    .start()
    .awaitTermination(5)			# block for 5 seconds, I am closing the session after 5 seconds for demo purpose, ideally it should run indefinitely to keep processing incoming data
)

26/01/09 14:02:05 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-5f29905b-509e-491a-94ef-69e4eded22ec. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/01/09 14:02:05 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/01/09 14:02:05 WARN AdminClientConfig: The configuration 'key.deserializer' was supplied but isn't a known config.
26/01/09 14:02:05 WARN AdminClientConfig: The configuration 'value.deserializer' was supplied but isn't a known config.
26/01/09 14:02:05 WARN AdminClientConfig: The configuration 'enable.auto.commit' was supplied but isn't a known config.
26/01/09 14:02:05 WARN AdminClientConfig: The configuration 'max.poll.records' was supplied but isn't a known con

-------------------------------------------
Batch: 0
-------------------------------------------
+----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+------+-----------------------+-------------+
|key |value                                                                                                                                                                                                                                                                        |topic      |partition|offset|timestamp              |timestampType|
+----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

False

In [11]:
# Schema of the Pyaload

from pyspark.sql.types import StringType, StructField, StructType, ArrayType, LongType

json_schema = (
    StructType(
    [StructField('customerId', StringType(), True),
    StructField('data', StructType(
        [StructField('devices',
                     ArrayType(StructType([
                        StructField('deviceId', StringType(), True),
                        StructField('measure', StringType(), True),
                        StructField('status', StringType(), True),
                        StructField('temperature', LongType(), True)
                    ]), True), True)
        ]), True),
    StructField('eventId', StringType(), True),
    StructField('eventOffset', LongType(), True),
    StructField('eventPublisher', StringType(), True),
    StructField('eventTime', StringType(), True)
    ])
)

In [12]:
# Apply the schema to payload to read the data
from pyspark.sql.functions import from_json,col

streaming_df = kafka_json_df.withColumn("values_json", from_json(col("value"), json_schema)).selectExpr("values_json.*")

In [ ]:
# To the schema of the data, place a sample json file and change readStream to read
streaming_df.printSchema()

root
 |-- customerId: string (nullable = true)
 |-- data: struct (nullable = true)
 |    |-- devices: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- deviceId: string (nullable = true)
 |    |    |    |-- measure: string (nullable = true)
 |    |    |    |-- status: string (nullable = true)
 |    |    |    |-- temperature: long (nullable = true)
 |-- eventId: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)



In [14]:
# Lets explode the data as devices contains list/array of device reading
from pyspark.sql.functions import explode

exploded_df = streaming_df.withColumn("data_devices", explode("data.devices"))

In [ ]:
# Check the schema of the exploded_df, place a sample json file and change readStream to read
exploded_df.printSchema()

root
 |-- customerId: string (nullable = true)
 |-- data: struct (nullable = true)
 |    |-- devices: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- deviceId: string (nullable = true)
 |    |    |    |-- measure: string (nullable = true)
 |    |    |    |-- status: string (nullable = true)
 |    |    |    |-- temperature: long (nullable = true)
 |-- eventId: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)
 |-- data_devices: struct (nullable = true)
 |    |-- deviceId: string (nullable = true)
 |    |-- measure: string (nullable = true)
 |    |-- status: string (nullable = true)
 |    |-- temperature: long (nullable = true)



In [16]:
# Flatten the exploded df
from pyspark.sql.functions import col

flattened_df = (
    exploded_df
    .drop("data")
    .withColumn("deviceId", col("data_devices.deviceId"))
    .withColumn("measure", col("data_devices.measure"))
    .withColumn("status", col("data_devices.status"))
    .withColumn("temperature", col("data_devices.temperature"))
    .drop("data_devices")
)

In [17]:
# Check the schema of the flattened_df, place a sample json file and change readStream to read
flattened_df.printSchema()

root
 |-- customerId: string (nullable = true)
 |-- eventId: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)
 |-- deviceId: string (nullable = true)
 |-- measure: string (nullable = true)
 |-- status: string (nullable = true)
 |-- temperature: long (nullable = true)



In [ ]:
# Write the output to console sink to check the output

(flattened_df
 .writeStream
 .format("console")
 .outputMode("append")
 .option("checkpointLocation", "checkpoint_dir_kafka")
 .start()
 .awaitTermination(5))  # block for 5 seconds, I am closing the session after 5 seconds for demo purpose, ideally it should run indefinitely to keep processing incoming data

26/01/09 14:00:03 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/01/09 14:00:03 WARN AdminClientConfig: The configuration 'key.deserializer' was supplied but isn't a known config.
26/01/09 14:00:03 WARN AdminClientConfig: The configuration 'value.deserializer' was supplied but isn't a known config.
26/01/09 14:00:03 WARN AdminClientConfig: The configuration 'enable.auto.commit' was supplied but isn't a known config.
26/01/09 14:00:03 WARN AdminClientConfig: The configuration 'max.poll.records' was supplied but isn't a known config.
26/01/09 14:00:03 WARN AdminClientConfig: The configuration 'auto.offset.reset' was supplied but isn't a known config.


-------------------------------------------
Batch: 0
-------------------------------------------
+----------+--------------------+-----------+--------------+--------------------+--------+-------+-------+-----------+
|customerId|             eventId|eventOffset|eventPublisher|           eventTime|deviceId|measure| status|temperature|
+----------+--------------------+-----------+--------------+--------------------+--------+-------+-------+-----------+
|   CI00117|7146c4a8-54ed-407...|      10012|        device|2023-01-05 11:13:...|    D002|      C|SUCCESS|          5|
+----------+--------------------+-----------+--------------+--------------------+--------+-------+-------+-----------+



False

In [ ]:
# Close the Spark session
spark.stop()

## Spark Streaming Triggers - Once, Processing Time & Continuous | Tune Kafka Streaming Performance

In [ ]:
# Create the Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Streaming from Kafka")
    .config("spark.streaming.stopGracefullyOnShutdown", True)
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0')
    .config("spark.sql.shuffle.partitions", 4)
    .master("local[*]")
    .getOrCreate()
)

spark

:: loading settings :: url = jar:file:/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e17b11c4-964c-4c10-acb9-dc4517a46b5a;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.3.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.3.0 in central
	found org.apache.kafka#kafka-clients;2.8.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.8.4 in central
	found org.slf4j#slf4j-api;1.7.32 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.2 in central
	found org.spark-project.spark#unused;1.0.0 in central
	found org.apache.hadoop#hadoop-client-api;3.3.2 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report 

26/01/09 22:58:36 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [2]:
# Create the kafka_df to read from kafka

kafka_df = (
    spark
    # .read
    .readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:19092")
    .option("subscribe", "device-data")
    .option("startingOffsets", "earliest")
    .load()
)

In [3]:
# View schema for raw kafka_df
kafka_df.printSchema()
# kafka_df.show() # show won't work on streaming data, it will work when the spark session is opened in read mode
# kafka_df.rdd.getNumPartitions()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [4]:
# Parse value from binay to string into kafka_json_df
from pyspark.sql.functions import expr

kafka_json_df = kafka_df.withColumn("value", expr("cast(value as string)"))
(
    kafka_json_df
    .writeStream
    .format("console")
    .outputMode("append")            # usually append for kafka sources
    .option("truncate", False)
    .start()
    .awaitTermination(5)			# block for 5 seconds, I am closing the session after 5 seconds for demo purpose, ideally it should run indefinitely to keep processing incoming data
)

26/01/10 17:18:11 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-b4575737-11f5-421d-bfa5-cfbaba11354d. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/01/10 17:18:11 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/01/10 17:18:12 WARN AdminClientConfig: The configuration 'key.deserializer' was supplied but isn't a known config.
26/01/10 17:18:12 WARN AdminClientConfig: The configuration 'value.deserializer' was supplied but isn't a known config.
26/01/10 17:18:12 WARN AdminClientConfig: The configuration 'enable.auto.commit' was supplied but isn't a known config.
26/01/10 17:18:12 WARN AdminClientConfig: The configuration 'max.poll.records' was supplied but isn't a known con

False

-------------------------------------------
Batch: 0
-------------------------------------------
+----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+------+-----------------------+-------------+
|key |value                                                                                                                                                                                                                                                                        |topic      |partition|offset|timestamp              |timestampType|
+----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [5]:
# Schema of the Pyaload

from pyspark.sql.types import StringType, StructField, StructType, ArrayType, LongType

json_schema = (
    StructType(
    [StructField('customerId', StringType(), True),
    StructField('data', StructType(
        [StructField('devices',
                     ArrayType(StructType([
                        StructField('deviceId', StringType(), True),
                        StructField('measure', StringType(), True),
                        StructField('status', StringType(), True),
                        StructField('temperature', LongType(), True)
                    ]), True), True)
        ]), True),
    StructField('eventId', StringType(), True),
    StructField('eventOffset', LongType(), True),
    StructField('eventPublisher', StringType(), True),
    StructField('eventTime', StringType(), True)
    ])
)

In [6]:
# Apply the schema to payload to read the data
from pyspark.sql.functions import from_json,col

streaming_df = kafka_json_df.withColumn("values_json", from_json(col("value"), json_schema)).selectExpr("values_json.*")

In [7]:
# To the schema of the data, place a sample json file and change readStream to read
streaming_df.printSchema()

root
 |-- customerId: string (nullable = true)
 |-- data: struct (nullable = true)
 |    |-- devices: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- deviceId: string (nullable = true)
 |    |    |    |-- measure: string (nullable = true)
 |    |    |    |-- status: string (nullable = true)
 |    |    |    |-- temperature: long (nullable = true)
 |-- eventId: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)



In [9]:
# Lets explode the data as devices contains list/array of device reading
from pyspark.sql.functions import explode

exploded_df = streaming_df.withColumn("data_devices", explode("data.devices"))

In [10]:
# Check the schema of the exploded_df, place a sample json file and change readStream to read
exploded_df.printSchema()

root
 |-- customerId: string (nullable = true)
 |-- data: struct (nullable = true)
 |    |-- devices: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- deviceId: string (nullable = true)
 |    |    |    |-- measure: string (nullable = true)
 |    |    |    |-- status: string (nullable = true)
 |    |    |    |-- temperature: long (nullable = true)
 |-- eventId: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)
 |-- data_devices: struct (nullable = true)
 |    |-- deviceId: string (nullable = true)
 |    |-- measure: string (nullable = true)
 |    |-- status: string (nullable = true)
 |    |-- temperature: long (nullable = true)



In [11]:
# Flatten the exploded df
from pyspark.sql.functions import col

flattened_df = (
    exploded_df
    .drop("data")
    .withColumn("deviceId", col("data_devices.deviceId"))
    .withColumn("measure", col("data_devices.measure"))
    .withColumn("status", col("data_devices.status"))
    .withColumn("temperature", col("data_devices.temperature"))
    .drop("data_devices")
)

In [12]:
# Check the schema of the flattened_df, place a sample json file and change readStream to read
flattened_df.printSchema()

root
 |-- customerId: string (nullable = true)
 |-- eventId: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)
 |-- deviceId: string (nullable = true)
 |-- measure: string (nullable = true)
 |-- status: string (nullable = true)
 |-- temperature: long (nullable = true)



In [ ]:
# Running in once/availableNow and processingTime mode
# Write the output to console sink to check the output

(flattened_df
 .writeStream
 .format("console")
 .outputMode("append")
#  .trigger(once=True)
 .trigger(processingTime='10 seconds')
 .option("checkpointLocation", "checkpoint_dir_kafka_1")
 .start()
 .awaitTermination())

26/01/10 17:24:33 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/01/10 17:24:33 WARN AdminClientConfig: The configuration 'key.deserializer' was supplied but isn't a known config.
26/01/10 17:24:33 WARN AdminClientConfig: The configuration 'value.deserializer' was supplied but isn't a known config.
26/01/10 17:24:33 WARN AdminClientConfig: The configuration 'enable.auto.commit' was supplied but isn't a known config.
26/01/10 17:24:33 WARN AdminClientConfig: The configuration 'max.poll.records' was supplied but isn't a known config.
26/01/10 17:24:33 WARN AdminClientConfig: The configuration 'auto.offset.reset' was supplied but isn't a known config.


False

In [ ]:
# Running in Continuous mode
# Write the output to memory sink to check the output

(kafka_df
 .writeStream
 .format("memory")
 .queryName("kafka_table")
 .outputMode("append")
 .trigger(continuous='10 seconds')
 .option("checkpointLocation", "checkpoint_dir_kafka_2")
 .start()
 .awaitTermination()
 )

In [ ]:
# View data from Memory Sink
spark.sql("select * from kafka_table").show()

In [ ]:
# Close the Spark session
spark.stop()

## Spark Streaming Writing data to Multiple Sinks | foreachBatch | Writing data to JDBC(Postgres)

Below code will download the required JAR file<br />
wget https://repo1.maven.org/maven2/org/postgresql/postgresql/42.2.20/postgresql-42.2.20.jar<br />
mkdir -p /home/jovyan/.ivy2/jars/<br />
mv postgresql-42.2.20.jar /home/jovyan/.ivy2/jars/org.postgresql_postgresql-42.2.20.jar<br /><br />

Run the below SQL Query to create the required tables in Postgres DB. Make sure to select connection as Postgres from the dropdown. You can access SQLPAD to query Postgres DB using https://localhost:3000.<br /> Login credentials for SQLPAD USER: admin@sqlpad.com PASSWORD: admin<br /><br />

CREATE TABLE public.device_data (<br />
	customerid varchar,<br />
	eventid varchar,<br />
	eventoffset varchar,<br />
	eventpublisher varchar,<br />
	eventtime varchar,<br />
	deviceid varchar,<br />
	measure varchar,<br />
	status varchar,<br />
	temperature varchar<br />
);<br />

In [ ]:
# Create the Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Writing to Multiple Sinks")
    .config("spark.streaming.stopGracefullyOnShutdown", True)
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0')
    .config('spark.jars', '/home/jovyan/.ivy2/jars/org.postgresql_postgresql-42.2.20.jar')
    .config("spark.sql.shuffle.partitions", 8)
    .master("local[*]")
    .getOrCreate()
)

spark

:: loading settings :: url = jar:file:/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e3f510e2-2e20-4267-be7b-341cbee4addf;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.3.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.3.0 in central
	found org.apache.kafka#kafka-clients;2.8.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.8.4 in central
	found org.slf4j#slf4j-api;1.7.32 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.2 in central
	found org.spark-project.spark#unused;1.0.0 in central
	found org.apache.hadoop#hadoop-client-api;3.3.2 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
downloading https://r

26/01/15 01:42:50 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [2]:
# Create the kafka_df to read from kafka

kafka_df = (
    spark
    # .read
    .readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:19092")
    .option("subscribe", "device-data")
    .option("startingOffsets", "earliest")
    .load()
)

In [3]:
# View schema for raw kafka_df
kafka_df.printSchema()
# kafka_df.show() # show won't work on streaming data, it will work when the spark session is opened in read mode
# kafka_df.rdd.getNumPartitions()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [4]:
# Parse value from binay to string into kafka_json_df
from pyspark.sql.functions import expr

kafka_json_df = kafka_df.withColumn("value", expr("cast(value as string)"))
(
    kafka_json_df
    .writeStream
    .format("console")
    .outputMode("append")            # usually append for kafka sources
    .option("truncate", False)
    .start()
    .awaitTermination(5)			# block for 5 seconds, I am closing the session after 5 seconds for demo purpose, ideally it should run indefinitely to keep processing incoming data
)

26/01/15 01:46:57 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-c624acbb-528a-4327-828c-bc34a8cf7854. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/01/15 01:46:57 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/01/15 01:47:00 WARN AdminClientConfig: The configuration 'key.deserializer' was supplied but isn't a known config.
26/01/15 01:47:00 WARN AdminClientConfig: The configuration 'value.deserializer' was supplied but isn't a known config.
26/01/15 01:47:00 WARN AdminClientConfig: The configuration 'enable.auto.commit' was supplied but isn't a known config.
26/01/15 01:47:00 WARN AdminClientConfig: The configuration 'max.poll.records' was supplied but isn't a known con

False

-------------------------------------------
Batch: 0
-------------------------------------------
+-------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+------+-----------------------+-------------+
|key                                                                                                          |value                                                                                                                                                                                                                                                                        |topic      |partition|offset|timestamp              |timestampType|
+----

In [5]:
# Schema of the Pyaload

from pyspark.sql.types import StringType, StructField, StructType, ArrayType, LongType

json_schema = (
    StructType(
    [StructField('customerId', StringType(), True),
    StructField('data', StructType(
        [StructField('devices',
                     ArrayType(StructType([
                        StructField('deviceId', StringType(), True),
                        StructField('measure', StringType(), True),
                        StructField('status', StringType(), True),
                        StructField('temperature', LongType(), True)
                    ]), True), True)
        ]), True),
    StructField('eventId', StringType(), True),
    StructField('eventOffset', LongType(), True),
    StructField('eventPublisher', StringType(), True),
    StructField('eventTime', StringType(), True)
    ])
)

In [6]:
# Apply the schema to payload to read the data
from pyspark.sql.functions import from_json,col

streaming_df = kafka_json_df.withColumn("values_json", from_json(col("value"), json_schema)).selectExpr("values_json.*")

In [7]:
# To the schema of the data, place a sample json file and change readStream to read
streaming_df.printSchema()

root
 |-- customerId: string (nullable = true)
 |-- data: struct (nullable = true)
 |    |-- devices: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- deviceId: string (nullable = true)
 |    |    |    |-- measure: string (nullable = true)
 |    |    |    |-- status: string (nullable = true)
 |    |    |    |-- temperature: long (nullable = true)
 |-- eventId: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)



In [9]:
# Lets explode the data as devices contains list/array of device reading
from pyspark.sql.functions import explode

exploded_df = streaming_df.withColumn("data_devices", explode("data.devices"))

In [10]:
# Check the schema of the exploded_df, place a sample json file and change readStream to read
exploded_df.printSchema()

root
 |-- customerId: string (nullable = true)
 |-- data: struct (nullable = true)
 |    |-- devices: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- deviceId: string (nullable = true)
 |    |    |    |-- measure: string (nullable = true)
 |    |    |    |-- status: string (nullable = true)
 |    |    |    |-- temperature: long (nullable = true)
 |-- eventId: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)
 |-- data_devices: struct (nullable = true)
 |    |-- deviceId: string (nullable = true)
 |    |-- measure: string (nullable = true)
 |    |-- status: string (nullable = true)
 |    |-- temperature: long (nullable = true)



In [11]:
# Flatten the exploded df
from pyspark.sql.functions import col

flattened_df = (
    exploded_df
    .drop("data")
    .withColumn("deviceId", col("data_devices.deviceId"))
    .withColumn("measure", col("data_devices.measure"))
    .withColumn("status", col("data_devices.status"))
    .withColumn("temperature", col("data_devices.temperature"))
    .drop("data_devices")
)

In [12]:
# Check the schema of the flattened_df, place a sample json file and change readStream to read
flattened_df.printSchema()

root
 |-- customerId: string (nullable = true)
 |-- eventId: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)
 |-- deviceId: string (nullable = true)
 |-- measure: string (nullable = true)
 |-- status: string (nullable = true)
 |-- temperature: long (nullable = true)



In [13]:
# Python function to write to multiple sinks
def device_data_output(df, batch_id):
    print("Batch id: "+ str(batch_id))

    # Write to parquet
    df.write.format("parquet").mode("append").save("output/device_data.parquet/")


    # Write to JDBC Postgres
    (
        df.write
        .mode("append")
        .format("jdbc")
        .option("driver", "org.postgresql.Driver")
        .option("url", "jdbc:postgresql://postgres-db:5432/sqlpad")
        .option("dbtable", "device_data")
        .option("user", "sqlpad")
        .option("password", "sqlpad")
        .save()
    )

    # Diplay
    df.show()

In [14]:
# Running foreachBatch
# Write the output to Multiple Sinks

(flattened_df
 .writeStream
 .foreachBatch(device_data_output)
 .trigger(processingTime='10 seconds')
 .option("checkpointLocation", "checkpoint_dir_kafka")
 .start()
 .awaitTermination())

26/01/15 02:07:25 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/01/15 02:07:25 WARN OffsetSeqMetadata: Updating the value of conf 'spark.sql.shuffle.partitions' in current session from '8' to '4'.
26/01/15 02:07:25 WARN AdminClientConfig: The configuration 'key.deserializer' was supplied but isn't a known config.
26/01/15 02:07:25 WARN AdminClientConfig: The configuration 'value.deserializer' was supplied but isn't a known config.
26/01/15 02:07:25 WARN AdminClientConfig: The configuration 'enable.auto.commit' was supplied but isn't a known config.
26/01/15 02:07:25 WARN AdminClientConfig: The configuration 'max.poll.records' was supplied but isn't a known config.
26/01/15 02:07:25 WARN AdminClientConfig: The configuration 'auto.offset.reset' was supplied but isn't a known config.


Batch id: 1


+----------+-------+-----------+--------------+---------+--------+-------+------+-----------+
|customerId|eventId|eventOffset|eventPublisher|eventTime|deviceId|measure|status|temperature|
+----------+-------+-----------+--------------+---------+--------+-------+------+-----------+
+----------+-------+-----------+--------------+---------+--------+-------+------+-----------+



26/01/15 02:07:36 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 10000 milliseconds, but spent 10838 milliseconds


-------------------------------------------
Batch: 1
-------------------------------------------
+----+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+------+-----------------------+-------------+
|key |value                                                                                                                                                                                                                                                                                                                                                                                                           

+----------+--------------------+-----------+--------------+--------------------+--------+-------+-------+-----------+
|customerId|             eventId|eventOffset|eventPublisher|           eventTime|deviceId|measure| status|temperature|
+----------+--------------------+-----------+--------------+--------------------+--------+-------+-------+-----------+
|   CI00104|cb8a6a8f-89c9-498...|      10017|        device|2023-01-05 11:13:...|    D004|      C|STANDBY|          5|
|   CI00104|cb8a6a8f-89c9-498...|      10017|        device|2023-01-05 11:13:...|    D004|      C|SUCCESS|         22|
|   CI00104|cb8a6a8f-89c9-498...|      10017|        device|2023-01-05 11:13:...|    D004|      C|  ERROR|          9|
+----------+--------------------+-----------+--------------+--------------------+--------+-------+-------+-----------+



ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
# Close the Spark session
spark.stop()

## Late Data Processing | Watermarks | Tumbling and Sliding Window Operations in Spark Streaming

In [1]:
# Create the Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Window Operations and Watermarks")
    .config("spark.streaming.stopGracefullyOnShutdown", True)
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0')
    .config("spark.sql.shuffle.partitions", 8)
    .master("local[*]")
    .getOrCreate()
)

spark

:: loading settings :: url = jar:file:/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c18c771a-88aa-48c9-bf8f-a9c2e3e34c8f;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.3.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.3.0 in central
	found org.apache.kafka#kafka-clients;2.8.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.8.4 in central
	found org.slf4j#slf4j-api;1.7.32 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.2 in central
	found org.spark-project.spark#unused;1.0.0 in central
	found org.apache.hadoop#hadoop-client-api;3.3.2 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
downloading https://r

In [ ]:
# Create the kafka_df to read from kafka

kafka_df = (
    spark
    .readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:19092")
    .option("subscribe", "wildlife")
    .option("startingOffsets", "earliest")
    .load()
)

26/01/22 11:19:40 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [3]:
# Convert binary to string value column
from pyspark.sql.functions import expr

kafka_json_df = kafka_df.withColumn("value", expr("cast(value as string)"))

In [4]:
from pyspark.sql.functions import from_json, col, split, explode

# JSON Schema
json_schema = "event_time string, data string"

# Expand JSON from Value column using Schema
json_df = kafka_json_df.withColumn("values_json", from_json(col("value"), json_schema))

In [5]:
# Select the required columns

flattened_df = json_df.select("values_json.event_time","values_json.data")

In [6]:
# Split the data in words

words_df = flattened_df \
    .withColumn("words", split("data", " ")) \
    .withColumn("word", explode("words")) \
    .withColumn("event_time", col("event_time").cast("timestamp"))

In [7]:
words_df.printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- data: string (nullable = true)
 |-- words: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- word: string (nullable = false)



In [ ]:
# Aggregate the words to generate count
from pyspark.sql.functions import count, lit, window

df_agg = words_df \
    .withWatermark("event_time", "10 minutes") \
    .groupBy(window("event_time", "10 minutes"),
    # .groupBy(window("event_time", "10 minutes", "5 minutes"), # sliding mode
                          "word").agg(count(lit(1)).alias("cnt"))

In [9]:
df_final = df_agg.selectExpr("window.start as start_time", "window.end as end_time", "word", "cnt")

In [10]:

df_final.printSchema()

root
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- word: string (nullable = false)
 |-- cnt: long (nullable = false)



In [ ]:
(df_final
 .writeStream
 .format("console")
 .outputMode("complete")
 .trigger(processingTime='30 seconds')
 .option("checkpointLocation", "checkpoint_dir_kafka_2")
 .start()
)

26/01/22 13:58:38 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/01/22 13:58:38 WARN StreamingQueryManager: Stopping existing streaming query [id=dde39712-d052-45c6-8367-e75e5de4c611, runId=349e3899-6806-4fa8-b9aa-d58c8c7bd16f], as a new run is being started.


26/01/22 13:58:38 WARN AdminClientConfig: The configuration 'key.deserializer' was supplied but isn't a known config.
26/01/22 13:58:38 WARN AdminClientConfig: The configuration 'value.deserializer' was supplied but isn't a known config.
26/01/22 13:58:38 WARN AdminClientConfig: The configuration 'enable.auto.commit' was supplied but isn't a known config.
26/01/22 13:58:38 WARN AdminClientConfig: The configuration 'max.poll.records' was supplied but isn't a known config.
26/01/22 13:58:38 WARN AdminClientConfig: The configuration 'auto.offset.reset' was supplied but isn't a known config.
26/01/22 13:58:39 WARN HDFSBackedStateStoreProvider: The state for version 6 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal for the first batch of starting query.
26/01/22 13:58:39 WARN HDFSBackedStateStoreProvider: The state for version 6 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal for 

-------------------------------------------
Batch: 6
-------------------------------------------
+-------------------+-------------------+----+---+
|         start_time|           end_time|word|cnt|
+-------------------+-------------------+----+---+
|2024-04-09 11:55:00|2024-04-09 12:05:00| dog|  1|
|2024-04-09 12:05:00|2024-04-09 12:15:00| dog|  2|
|2024-04-09 11:55:00|2024-04-09 12:05:00| owl|  3|
|2024-04-09 12:00:00|2024-04-09 12:10:00| owl|  4|
|2024-04-09 12:05:00|2024-04-09 12:15:00| owl|  2|
|2024-04-09 12:15:00|2024-04-09 12:25:00| dog|  1|
|2024-04-09 10:55:00|2024-04-09 11:05:00| dog|  1|
|2024-04-09 12:10:00|2024-04-09 12:20:00| dog|  2|
|2024-04-09 12:10:00|2024-04-09 12:20:00| owl|  1|
|2024-04-09 11:00:00|2024-04-09 11:10:00| dog|  1|
|2024-04-09 12:00:00|2024-04-09 12:10:00| dog|  2|
+-------------------+-------------------+----+---+



In [ ]:
(df_final
 .writeStream
 .format("console")
 .outputMode("update")
 .trigger(processingTime='30 seconds')
 .option("checkpointLocation", "checkpoint_dir_kafka_3")
 .start()
)

26/01/22 13:58:42 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/01/22 13:58:42 WARN StreamingQueryManager: Stopping existing streaming query [id=fc4edb9c-ece9-43cc-8cde-1324a8a1c614, runId=07a371d8-4407-4136-98cd-e2df3489e188], as a new run is being started.


26/01/22 13:58:42 WARN AdminClientConfig: The configuration 'key.deserializer' was supplied but isn't a known config.
26/01/22 13:58:42 WARN AdminClientConfig: The configuration 'value.deserializer' was supplied but isn't a known config.
26/01/22 13:58:42 WARN AdminClientConfig: The configuration 'enable.auto.commit' was supplied but isn't a known config.
26/01/22 13:58:42 WARN AdminClientConfig: The configuration 'max.poll.records' was supplied but isn't a known config.
26/01/22 13:58:42 WARN AdminClientConfig: The configuration 'auto.offset.reset' was supplied but isn't a known config.
26/01/22 13:58:43 WARN HDFSBackedStateStoreProvider: The state for version 11 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal for the first batch of starting query.
26/01/22 13:58:43 WARN HDFSBackedStateStoreProvider: The state for version 11 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal fo

-------------------------------------------
Batch: 11
-------------------------------------------
+-------------------+-------------------+----+---+
|         start_time|           end_time|word|cnt|
+-------------------+-------------------+----+---+
|2024-04-09 12:05:00|2024-04-09 12:15:00| dog|  2|
|2024-04-09 12:00:00|2024-04-09 12:10:00| dog|  2|
+-------------------+-------------------+----+---+



In [ ]:
# Close the Spark session
spark.stop()

26/01/22 14:01:50 WARN StateStore: Error running maintenance thread
java.lang.IllegalStateException: SparkEnv not active, cannot do maintenance on StateStores
	at org.apache.spark.sql.execution.streaming.state.StateStore$.doMaintenance(StateStore.scala:632)
	at org.apache.spark.sql.execution.streaming.state.StateStore$.$anonfun$startMaintenanceIfNeeded$1(StateStore.scala:610)
	at org.apache.spark.sql.execution.streaming.state.StateStore$MaintenanceTask$$anon$1.run(StateStore.scala:453)
	at java.base/java.util.concurrent.Executors$RunnableAdapter.call(Executors.java:572)
	at java.base/java.util.concurrent.FutureTask.runAndReset(FutureTask.java:358)
	at java.base/java.util.concurrent.ScheduledThreadPoolExecutor$ScheduledFutureTask.run(ScheduledThreadPoolExecutor.java:305)
	at java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.java:1144)
	at java.base/java.util.concurrent.ThreadPoolExecutor$Worker.run(ThreadPoolExecutor.java:642)
	at java.base/java.lang.Thread.

## Read and Write from Azure Cosmos DB using Spark | E2E Cosmos DB setup | NoSQL vs SQL Databases

ACID: Atomicity, Consistency, Isolation, Durability

In [32]:
# Create the Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Read and Write using Cosmos DB")
    .config("spark.streaming.stopGracefullyOnShutdown", True)
    .config('spark.jars.packages', 'com.azure.cosmos.spark:azure-cosmos-spark_3-3_2-12:4.15.0')
    .config("spark.sql.shuffle.partitions", 8)
    .master("local[*]")
    .getOrCreate()
)

spark

In [ ]:
# Set configuration settings to connect to Cosmos DB

config = {
  "spark.cosmos.accountEndpoint": "<cosmos-db-endpoint>",
  "spark.cosmos.accountKey": "<secret-key>",
  "spark.cosmos.database": "easewithdata",
  "spark.cosmos.container": "device-data"
}

# put the endpoint and key in the conf file in spark's directory. Which is at /usr/local/spark/conf/spark-defaults.conf

In [ ]:
# Read data from Cosmos DB

df = (
    spark.read.format("cosmos.oltp")
    .options(**config)
    .option("spark.cosmos.read.inferSchema.enabled", "true")
    .load()

)

In [ ]:
df.show()

In [ ]:
# Write data to Cosmos DB

df_read = spark.read.json("input/device_files/device_03.json")

In [ ]:
df_read.show()

In [ ]:
# Write data to Cosmos DB
from pyspark.sql.functions import col

df_read.withColumn("id", col("eventId")).write \
    .format("cosmos.oltp") \
    .options(**config) \
    .option("spark.cosmos.write.strategy", "ItemAppend") \
    .mode("APPEND") \
    .save()
    # .option("spark.cosmos.write.strategy", "ItemOverwrite") \
    # .option("spark.cosmos.write.strategy", "ItemDelete") \

In [ ]:
# Close the Spark session
spark.stop()